# Pratica — Lezione 5: confrontare gli inizializzatori nella self-attention

## Contesto

Nel notebook di lezione hai addestrato un mini traduttore EN→IT il cui layer
`SelfAttention` inizializza le sue matrici `Wq`, `Wk`, `Wv` con l'inizializzatore
`glorot_uniform` (Xavier). Questo è l'**esercizio reale** proposto nel
materiale sorgente (notebook "Transformers with Keras" di IBM, esercizio 1):
addestrare la stessa architettura ma cambiando l'inizializzatore in
`he_uniform`, e confrontare le curve di perdita dei due addestramenti in uno
stesso grafico.

Gli inizializzatori decidono con quali valori casuali partono i pesi
prima dell'addestramento. Glorot (`glorot_uniform`) è pensato per attivazioni
simmetriche come `tanh`; He (`he_uniform`) è pensato per `relu` e
varianti. Qui non usiamo `relu` nel layer di attenzione, quindi questo
esercizio ti serve anche a verificare empiricamente se il cambio è rilevante
o meno in questo caso concreto.

## Passo 0: installare le dipendenze e preparare i dati (come nella lezione)

In [ ]:
!pip install tensorflow_cpu==2.17.1
!pip install matplotlib==3.9.2

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, Layer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import backend as K
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter('ignore', FutureWarning)

In [ ]:
# Stessi dati del notebook di lezione
input_texts = [
    "Hello.", "How are you?", "I am learning machine translation.",
    "What is your name?", "I love programming."
]
target_texts = [
    "Ciao.", "Come stai?", "Sto imparando la traduzione automatica.",
    "Come ti chiami?", "Amo programmare."
]
target_texts = ["startseq " + x + " endseq" for x in target_texts]

input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

output_tokenizer = Tokenizer()
output_tokenizer.fit_on_texts(target_texts)
output_sequences = output_tokenizer.texts_to_sequences(target_texts)

input_vocab_size = len(input_tokenizer.word_index) + 1
output_vocab_size = len(output_tokenizer.word_index) + 1

max_input_length = max([len(seq) for seq in input_sequences])
max_output_length = max([len(seq) for seq in output_sequences])

input_sequences = pad_sequences(input_sequences, maxlen=max_input_length, padding='post')
output_sequences = pad_sequences(output_sequences, maxlen=max_output_length, padding='post')

decoder_input_data = output_sequences[:, :-1]
decoder_output_data = output_sequences[:, 1:]
decoder_output_data = np.array([np.eye(output_vocab_size)[seq] for seq in decoder_output_data])

## Impalcatura: layer di attenzione e architettura parametrizzabili

Per non ripetere l'architettura completa due volte, questa versione di
`SelfAttention` riceve il nome dell'inizializzatore come parametro, e
`build_model(initializer)` costruisce il modello completo (encoder + decoder +
attenzione) usando quell'inizializzatore. È esattamente lo stesso modello della
lezione — solo che ora puoi riutilizzarlo con inizializzatori diversi senza
copiare e incollare codice.

In [ ]:
class SelfAttention(Layer):
    def __init__(self, initializer='glorot_uniform', **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.initializer = initializer

    def build(self, input_shape):
        feature_dim = input_shape[0][-1]

        self.Wq = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer=self.initializer,
            trainable=True,
            name='Wq'
        )
        self.Wk = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer=self.initializer,
            trainable=True,
            name='Wk'
        )
        self.Wv = self.add_weight(
            shape=(feature_dim, feature_dim),
            initializer=self.initializer,
            trainable=True,
            name='Wv'
        )
        super(SelfAttention, self).build(input_shape)

    def call(self, inputs):
        q, k, v = inputs
        q = K.dot(q, self.Wq)
        k = K.dot(k, self.Wk)
        v = K.dot(v, self.Wv)

        scores = K.batch_dot(q, k, axes=[2, 2])
        dk = K.cast(K.shape(k)[-1], dtype=K.floatx())
        scores = scores / K.sqrt(dk)

        attention_weights = K.softmax(scores, axis=-1)
        output = K.batch_dot(attention_weights, v)
        return output


def build_model(initializer='glorot_uniform'):
    """Costruisce e compila il modello encoder-decoder + self-attention,
    usando `initializer` per le matrici Wq, Wk, Wv del layer di attenzione."""
    encoder_inputs = Input(shape=(max_input_length,))
    encoder_embedding = Embedding(input_vocab_size, 256)(encoder_inputs)
    encoder_lstm = LSTM(256, return_sequences=True, return_state=True)
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
    encoder_states = [state_h, state_c]

    decoder_inputs = Input(shape=(max_output_length - 1,))
    decoder_embedding = Embedding(output_vocab_size, 256)(decoder_inputs)
    decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
    decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

    self_attention = SelfAttention(initializer=initializer)
    attention_output = self_attention([decoder_outputs, encoder_outputs, encoder_outputs])

    decoder_concat = Concatenate(axis=-1)([decoder_outputs, attention_output])
    decoder_dense = Dense(output_vocab_size, activation='softmax')
    decoder_outputs = decoder_dense(decoder_concat)

    model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## Modello base: `glorot_uniform` (già addestrato per te)

Questa cella addestra il modello di riferimento, come nel notebook di
lezione, e salva il suo storico in `history_glorot`.

In [ ]:
model_glorot = build_model(initializer='glorot_uniform')

history_glorot = model_glorot.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16,
    verbose=0
)

print("Perdita finale (glorot_uniform):", history_glorot.history['loss'][-1])

## Esercizio: addestra con `he_uniform` e confronta

**Consegna**: crea un secondo modello identico a quello sopra, ma usando
`initializer='he_uniform'` nel layer di self-attention. Addestralo con gli
stessi iperparametri (100 epoche, batch_size=16), salva il suo storico in
`history_he`, e rappresenta graficamente entrambe le curve di perdita
(`history_glorot` vs `history_he`) nello stesso grafico per confrontare quale
converge più velocemente o più in basso.

Suggerimenti:
- Hai già a disposizione la funzione `build_model(initializer=...)` da riutilizzare.
- `history.history['loss']` è la lista dei valori di perdita per epoca.
- Usa `plt.plot(..., label=...)` e `plt.legend()` per distinguere le curve.

In [ ]:
# TODO: costruisci un secondo modello usando initializer='he_uniform'
model_he = None  # TODO: usa build_model('he_uniform')

# TODO: addestra il modello con gli stessi iperparametri del modello base
# (epochs=100, batch_size=16) e salva il risultato in history_he
history_he = None

# TODO: rappresenta graficamente history_glorot.history['loss'] e history_he.history['loss']
# nello stesso grafico, con legenda, titolo ed etichette degli assi

<details>
<summary>💡 Vedi soluzione</summary>

```python
# Costruiamo il secondo modello con he_uniform
model_he = build_model(initializer='he_uniform')

# Lo addestriamo con gli stessi iperparametri del modello base
history_he = model_he.fit(
    [input_sequences, decoder_input_data],
    decoder_output_data,
    epochs=100,
    batch_size=16,
    verbose=0
)

print("Perdita finale (he_uniform):", history_he.history['loss'][-1])

# Confrontiamo entrambe le curve di perdita nello stesso grafico
plt.plot(history_glorot.history['loss'], label='glorot_uniform', color='red')
plt.plot(history_he.history['loss'], label='he_uniform', color='blue')
plt.title('Perdita di addestramento: glorot_uniform vs he_uniform')
plt.xlabel('Epoche')
plt.ylabel('Perdita')
plt.legend()
plt.show()
```

Con un dataset di sole 5 frasi, è normale che entrambe le curve convergano a
una perdita molto bassa e che la differenza tra gli inizializzatori sia
piccola o incoerente tra le esecuzioni (i pesi partono casuali). Il valore
dell'esercizio sta nel processo — parametrizzare il modello, addestrare due
varianti e confrontare le curve — più che in un vincitore netto.
</details>

## Per continuare

- Torna al notebook di lezione:
  [clase05_transformers_atencion.ipynb](../notebooks_clase/clase05_transformers_atencion_IT.ipynb).
- Ripassa la teoria in
  [manual/08_transformers_atencion.md](../manual/08_transformers_atencion_IT.md).
- Se vuoi andare oltre, l'esercizio 2 del materiale originale propone di
  confrontare l'ottimizzatore `adam` con `adagrad` seguendo lo stesso schema
  che hai usato qui per gli inizializzatori.